# 05 · One-stage, anchor-free: YOLO11

**Model.** Ultralytics YOLO11s, COCO-pretrained: anchor-free decoupled head at strides 8/16/32,
task-aligned assignment (TAL), BCE classification + CIoU and distribution-focal (DFL) box losses.
**Protocol.** Same split, frames and validation as notebooks 03–04. Ultralytics' default augmentation
(mosaic, scale ±50 %, HSV, flip); SGD lr 0.01; **2 GPU-hours per run** (`time: 1.0` on 2× T4). Ultralytics
adapts the epoch count to the time budget, so every run here and in notebook 06 gets the same compute.

| run | change | search range / motivation |
|---|---|---|
| `yolo11s_base` | imgsz 1280 | reference (benchmark resolution) |
| `yolo11s_960`, `yolo11s_1536` | input resolution | {960, **1280**, 1536}; objects scale linearly with input size |
| `yolo11s_p2` | extra stride-4 (P2) head | 10 % of objects < 16 px: at stride 8 they cover ~2 grid points |
| `yolo11s_tal` | TAL top-k 13 instead of 10 (assignment) | more positives for small objects in dense scenes |

## How locations are assigned to objects without anchors

Every cell of each output grid (stride *s* ∈ {8, 16, 32}) is an **anchor point** that predicts class
scores and the distances from the point to the four box sides (as a discrete distribution, DFL). For each
ground-truth box, TAL:
1. keeps the candidate points that lie **inside the box**,
2. scores each candidate by the alignment metric *t = s^α · IoU^β* (α = 0.5, β = 6), where *s* is the
   predicted score for the object's class and IoU is between the point's predicted box and the object,
3. takes the **top-k (10)** candidates as positives, with soft targets ∝ *t*. When a point is claimed by
   several objects, it goes to the one with the highest IoU.

**Effect on small and dense objects.** A 12×12 px object contains about 2×2 points at stride 8 and about one
at stride 16, so it can never get 10 positives. Its candidates are also shared with neighbours in a crowd,
where conflicts are resolved toward the larger-IoU, usually larger, object. Raising the input resolution
or adding a stride-4 level (P2) multiplies the points inside small boxes. A larger top-k gives each object
more positives, at the risk of low-quality ones.

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "ivan-in-a-nutshell/AI_HW_2"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Data and the fixed benchmark protocol: fold 0, every 10th frame, unmasked test-scale validation frames
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from visdet.data import find_raw_root, find_prepared, read_prepared, fold_split
from visdet.evaluate import build_gt
from visdet.plotting import setup_style, save, SERIES
from visdet.results import load_log, family_table, best_run, plot_histories, plot_breakdown, styled

RAW, PREP = find_raw_root(), find_prepared()
RUNS = Path("/kaggle/working/runs") if ON_KAGGLE else REPO / "runs"
FOLD, STEP = 0, 10
images, boxes, folds = read_prepared(PREP)
train_rows, val_rows = fold_split(images, folds, FOLD, train_step=STEP, val_step=STEP)
gt = build_gt(val_rows, boxes)
N_GPU = torch.cuda.device_count()
SMOKE = os.environ.get("VISDET_SMOKE") == "1"   # CPU dry run: 1 epoch on a few frames, nothing logged for real
if SMOKE:
    val_rows = val_rows.head(8); gt = build_gt(val_rows, boxes)
setup_style()
print(f"fold {FOLD}: {len(train_rows)} training frames, {len(val_rows)} validation frames; {N_GPU} GPU(s)")

In [ ]:
# Ultralytics dataset files for fold 0 (lists of prepared, masked frames)
from visdet.ul_utils import write_yolo_data, train_ultralytics, evaluate_ultralytics
DATA_YAML = write_yolo_data(PREP, images, folds, FOLD, RUNS / "data_fold0", train_step=STEP, val_step=STEP,
                            val_limit=16 if SMOKE else None)
DEVICE = ",".join(str(i) for i in range(N_GPU)) or "cpu"
TRAIN_KW = {"device": DEVICE, "workers": 4, "verbose": False}
UL_SMOKE = {"imgsz": 320, "epochs": 1, "time": None, "batch": 4, "fraction": 0.005, "workers": 0} if SMOKE else {}

## Training and evaluation

In [ ]:
# Train each configuration on the 2 GPU-hour budget, then evaluate with the shared protocol
CONFIGS = ['yolo11s_base', 'yolo11s_960', 'yolo11s_1536', 'yolo11s_p2', 'yolo11s_tal']
for name in CONFIGS:
    weights = RUNS / name / "weights" / "best.pt"
    if not weights.exists():
        weights, cfg = train_ultralytics(f"configs/{name}.yaml", DATA_YAML, RUNS, **(TRAIN_KW | UL_SMOKE))
        evaluate_ultralytics(weights, cfg, RAW, val_rows, boxes, fold=FOLD, **({"imgsz": 320} if SMOKE else {}))

## Results

In [ ]:
# All YOLO runs; then initial vs tuned YOLO against the tuned anchor-based detectors
log = load_log()
display(styled(family_table(log, CONFIGS)))
ANCHOR_BASED = [best_run(log, fam) for fam in (['frcnn_base', 'frcnn_anchors', 'frcnn_proposals', 'frcnn_giou'], ['retina_base', 'retina_anchors', 'retina_iou', 'retina_focal']) if any(r in log.index for r in fam)]
compare = [CONFIGS[0], best_run(log, CONFIGS)] + ANCHOR_BASED
styled(family_table(log, compare))

In [ ]:
# AP by object size and by scene density: YOLO (initial, tuned) vs the anchor-based detectors
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.4))
plot_breakdown(log, compare, ax=a, title="AP by object size")
dens = [c for c in log.columns if "dens" in c]
plot_breakdown(log, compare, cols=dens, labels=[c.replace("AP_dens", "") + " objects" for c in dens], ax=b,
               title="AP by objects per frame")
fig.tight_layout(); save(fig, "size_density_breakdown")

In [ ]:
# Learning curves of the YOLO runs (Ultralytics validator on masked frames) against GPU time
ax = plot_histories([c for c in CONFIGS if c in log.index], x="gpu_hours", title="YOLO11s: validation mAP vs GPU time")
save(ax.figure, "yolo_curves")

## Failure analysis

Confident detections (score ≥ 0.3) of the best run are labelled with an error type and every object
as found / missed (`visdet.analysis`): background false positives, wrong class, poor localisation,
duplicates, and misses broken down by size and occlusion.

In [ ]:
# Predict the validation frames with the best run and label every error
from visdet.analysis import classify_errors, error_summary, confusion, frame_errors, draw_errors
from visdet.infer import predict_eval_frames, load_frame
from visdet.infer import UltralyticsPredictor
from visdet.config import load_config
BEST = best_run(log, CONFIGS)
predictor = UltralyticsPredictor(RUNS / BEST / "weights" / "best.pt", imgsz=load_config(f"configs/{BEST}.yaml")["imgsz"])
dets = predict_eval_frames(predictor, RAW, val_rows, desc="validation")
det_t, gt_t = classify_errors(boxes[boxes.image_id.isin(val_rows.image_id)], dets, score_thr=0.3)
shares, recall = error_summary(det_t, gt_t)
display(shares.to_frame("share of confident detections").T.round(3))
display(recall.round(3))
confusion(det_t)

In [ ]:
# Three failure examples: most missed objects, most class confusions, most background false positives
fe = frame_errors(det_t, gt_t)
picks = {"missed objects": fe.missed.idxmax(),
          "class confusion": fe.get("wrong_cls", fe.missed).idxmax(),
          "background false positives": fe.get("background", fe.missed).idxmax()}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (what, img_id) in zip(axes, picks.items()):
    r = val_rows.set_index("image_id").loc[img_id]
    frame = load_frame(RAW / r.file_name, (r.out_w, r.out_h))
    vis = draw_errors(frame, gt_t[gt_t.image_id == img_id], det_t[det_t.image_id == img_id])
    ax.imshow(vis[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"{what}: {r['clip']} #{r.frame} ({fe.loc[img_id, 'missed']}/{fe.loc[img_id, 'objects']} missed)", fontsize=8)
fig.suptitle("green = found, yellow = missed, red = background FP, orange = wrong class, magenta = poor localisation", fontsize=8)
fig.tight_layout()
save(fig, "yolo_failures", photo=True)

## Submission from the best YOLO run

The first leaderboard submission (Phase 0) comes from here: inference on the original test frames, at
most 500 detections per image, boxes in original pixels.

In [ ]:
# Predict the test set, validate the format and write submission.csv
from visdet.infer import UltralyticsPredictor
from visdet.submission import predict_test, make_submission, validate_submission
from visdet.config import load_config
BEST = best_run(log, CONFIGS)
cfg = load_config(f"configs/{BEST}.yaml")
predictor = UltralyticsPredictor(RUNS / BEST / "weights" / "best.pt", imgsz=cfg["imgsz"], max_det=500)
test = pd.read_csv(RAW / "test_images.csv")
sub = make_submission(predict_test(predictor, RAW, limit=16 if SMOKE else None), test)
validate_submission(sub, test)
sub.to_csv("/kaggle/working/submission.csv" if ON_KAGGLE else "submission.csv", index=False)
print(BEST, "->", (sub.PredictionString.str.split().str.len().fillna(0) // 6).describe()[["mean", "max"]].to_dict())

## Findings

_Fill in after the runs:_ the selected resolution and why (AP_s vs FPS), whether P2 and the larger TAL
top-k help small/dense objects, how many epochs each run reached in the budget, and how the initial and
tuned YOLO compare with the anchor-based detectors per object size.

**After the session:** download `/kaggle/working/repo/experiments/` (metrics, per-epoch histories) and
`images/` from the notebook output and commit them, so later notebooks and the report can use them.
Checkpoints stay in the notebook output (`/kaggle/working/runs`).